**Achieved Training Accuracy of 0.8472 (84.72%) after 100 epochs and Evaluation Accuracy of 0.8491 (84.91%)**

In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.layers import (
    Conv2D, MaxPool2D, Dense, Rescaling, Flatten, Dropout,
    RandomFlip, RandomZoom, RandomRotation, RandomTranslation,
    Input, BatchNormalization, GlobalAveragePooling2D
)
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
import matplotlib.pyplot as plt


In [ ]:
(X_train , Y_train) , (X_test , Y_test) = cifar10.load_data()

In [ ]:
print(X_train.shape)
print(Y_train.shape)

In [ ]:
Y_train = Y_train.reshape(-1)
print(Y_train.shape)

Y_test = Y_test.reshape(-1)
print(Y_test.shape)

In [ ]:
classes = [
    'airplane',
    'automobile',
    'bird',
    'cat',
    'deer',
    'dog',
    'frog',
    'horse',
    'ship',
    'truck'
]

In [ ]:
def plot_sample(X , Y , index) :
  plt.figure(figsize = (15,2))
  plt.imshow(X[index])
  plt.xlabel(classes[Y[index]])

In [ ]:
plot_sample(X_train,Y_train , 1)

In [ ]:
normalizer = Rescaling(1/255)

X_train = normalizer(X_train)
X_test = normalizer(X_test)

**When designing the convolutional layers of a CNN, a good standard practice is to start with a modest number of filters in the early layers and progressively double them as you go deeper into the network (e.g., 32 → 64 → 128 → 256).**

In [ ]:
#Data Augmentation
data_augmentation = Sequential([
    RandomFlip("horizontal"),
    RandomTranslation(0.1, 0.1),
    RandomZoom(0.1),
    RandomRotation(0.08),
])


In [ ]:
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=20,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_accuracy',
    factor=0.5,
    patience=6,
    min_lr=1e-6,
    verbose=1
)


In [ ]:
#Sequential needs layers to be passed into as a List
#filters means how many features should be extracted by the kernel when it strides over the image
weight_decay = 1e-4

def conv_block(filters):
    return [
        Conv2D(filters, (3, 3), padding='same', kernel_regularizer=l2(weight_decay)),
        BatchNormalization(),
        tf.keras.layers.Activation('relu'),
        Conv2D(filters, (3, 3), padding='same', kernel_regularizer=l2(weight_decay)),
        BatchNormalization(),
        tf.keras.layers.Activation('relu'),
        MaxPool2D((2, 2)),
    ]

cnn_model = Sequential([
    Input(shape=(32, 32, 3)),
    data_augmentation,

    *conv_block(32),
    Dropout(0.2),

    *conv_block(64),
    Dropout(0.3),

    *conv_block(128),
    Dropout(0.4),

    GlobalAveragePooling2D(),
    Dense(256, activation='relu', kernel_regularizer=l2(weight_decay)),
    BatchNormalization(),
    Dropout(0.5),
    Dense(10, activation='softmax')
])


In [ ]:
cnn_model.summary()

If instead your labels look like: [0,0,0,0,0,0,1,0,0,0]  which is 'one-hot encoded'

then use: loss='categorical_crossentropy'

CIFAR-10 normally gives labels like:

0
1
6
3
9
...

Then use:
loss='sparse_categorical_crossentropy'

In [ ]:
cnn_model.compile(
    optimizer = 'adam',
    loss = 'sparse_categorical_crossentropy',
    metrics = ['accuracy']
)

**verbose in model.fit() controls how much information Keras prints in the output while training.**

verbose=1 — progress bar

This is the default in many Keras setups.
You'll see something like:

Epoch 1/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step
loss: 1.42 - accuracy: 0.48

It shows a progress bar as batches are processed.



verbose=0 — completely silent
cnn_model.fit(
    X_train,
    Y_train,
    epochs=10,
    verbose=0
)

Nothing is printed while training.
Useful when you don't want your notebook flooded with output.



verbose=2 — one line per epoch
cnn_model.fit(
    X_train,
    Y_train,
    epochs=10,
    verbose=2
)

Instead of a live progress bar, you get something like:

Epoch 1/10
loss: 1.42 - accuracy: 0.48
Epoch 2/10
loss: 1.18 - accuracy: 0.57
Epoch 3/10
loss: 1.02 - accuracy: 0.64




In [ ]:
history = cnn_model.fit(
    X_train, Y_train,
    epochs=200,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'], label='train_accuracy')
axes[0].plot(history.history['val_accuracy'], label='val_accuracy')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'], label='train_loss')
axes[1].plot(history.history['val_loss'], label='val_loss')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Loss')
axes[1].set_title('Loss')
axes[1].legend()

plt.show()


In [ ]:
cnn_model.evaluate(X_test , Y_test) #"How good is my model?"

In [ ]:
Y_pred = cnn_model.predict(X_test) #"What does my model predict for each individual image?"
print(Y_pred[:5])

**For a 2D NumPy array:**

**axis=0 means ↓ down**

**axis=1 means → across**

In [ ]:
import numpy as np
Y_classes = np.argmax(Y_pred , axis = 1)

In [ ]:
print(Y_classes[0:17])
print(Y_test[0:17])

In [ ]:
plot_sample(X_test, Y_test, 12)

In [ ]:
plot_sample(X_test, Y_classes, 12)

**Save this Model**

In [ ]:
cnn_model.save('cifar10_model.keras') #temporarily in colab's runtime

In [ ]:
# Save the model (Kaggle: everything in /kaggle/working/ is saved with the notebook version
# and can be downloaded from the Output tab after you commit/save)
import os

save_dir = '/kaggle/working'
os.makedirs(save_dir, exist_ok=True)

cnn_model.save(f'{save_dir}/cifar10_model.keras')


**To load the model**

In [ ]:
# from tensorflow.keras.models import load_model

# On Kaggle, load it back from /kaggle/working/ (same session) or from wherever
# you attached the saved model as an input dataset:
# cnn_model = load_model('/kaggle/working/cifar10_model.keras')
